In [1]:
#!/usr/bin/env python3
import os
import sys
import json
from pathlib import Path
from tqdm import tqdm

from lerobot_so_arm.config import get_path
from lerobot_so_arm.utils.transform import DatasetTransformer
from lerobot_so_arm.utils.data_loading import load_so_episode

vjepa_root = get_path('vjepa_root')
sys.path.append(vjepa_root)

SOURCE_FOLDER = get_path('datasets') + '/source'
OUTPUT_FOLDER = get_path('datasets') + '/output'
DATASET_LIST_FILE = 'dataset_list.txt'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Initialize the dataset transformer
transformer = DatasetTransformer()
print("DatasetTransformer initialized with automatic calibration detection")

DatasetTransformer initialized with automatic calibration detection


In [2]:
# Read dataset list
dataset_list_path = os.path.join(SOURCE_FOLDER, DATASET_LIST_FILE)

if not os.path.exists(dataset_list_path):
    print(f"ERROR: Dataset list file not found at {dataset_list_path}")
    print("Please ensure the source folder contains dataset_list.txt")
else:
    # Read the dataset list
    with open(dataset_list_path, 'r') as f:
        relative_paths = [line.strip() for line in f.readlines() if line.strip()]
    
    # Convert relative paths to absolute paths by prepending source folder
    dataset_paths = [os.path.join(SOURCE_FOLDER, path) for path in relative_paths]
    
    print(f"Found {len(dataset_paths)} episodes in dataset list:")
    for i, path in enumerate(dataset_paths[:5]):  # Show first 5 episodes
        print(f"  {i+1}. {path}")
    
    if len(dataset_paths) > 5:
        print(f"  ... and {len(dataset_paths) - 5} more episodes")

Found 3 episodes in dataset list:
  1. /Users/michelmeyer/.local/dev/test_dataset/source/episodes/smanni+train_so100_all-episode_001
  2. /Users/michelmeyer/.local/dev/test_dataset/source/episodes/smanni+train_so100_all-episode_002
  3. /Users/michelmeyer/.local/dev/test_dataset/source/episodes/smanni+train_so100_all-episode_003


In [8]:
# Process all episodes using the shared DatasetTransformer
if 'dataset_paths' in locals() and len(dataset_paths) > 0:
    print(f"Processing {len(dataset_paths)} episodes using DatasetTransformer...")
    
    results = {'successful': 0, 'failed': 0, 'failed_episodes': []}
    
    for index, episode_path in enumerate(dataset_paths, 1):
        try:
            episode_name = os.path.basename(episode_path)
            output_path = os.path.join(OUTPUT_FOLDER, episode_name)
            print(f">>> [{index}/{len(dataset_paths)}] Processing: {episode_name}")
            transformer.transform_episode(episode_path, output_path)
            results['successful'] += 1
            
        except Exception as e:
            results['failed'] += 1
            results['failed_episodes'].append(episode_path)
            print(f"✗ Failed: {os.path.basename(episode_path)}: {e}")
    
    # Print summary
    print(f"\nPROCESSING SUMMARY")
    print(f"Total episodes: {len(dataset_paths)}")
    print(f"Successful: {results['successful']}")
    print(f"Failed: {results['failed']}")
    
    if results['failed_episodes']:
        print(f"Failed episodes:")
        for failed_ep in results['failed_episodes']:
            print(f"  - {os.path.basename(failed_ep)}")
    
else:
    print("No dataset paths available for batch processing")


Processing 3 episodes using DatasetTransformer...
>>> [1/3] Processing: smanni+train_so100_all-episode_001
Auto-detected calibration: so_old_calibration
>>> [2/3] Processing: smanni+train_so100_all-episode_002
Auto-detected calibration: so_old_calibration
>>> [3/3] Processing: smanni+train_so100_all-episode_003
Auto-detected calibration: so_old_calibration

PROCESSING SUMMARY
Total episodes: 3
Successful: 3
Failed: 0
